# Twin-2K-500 — load and inspect (see `DATA.md`)

## 1. Download catalog and human response CSVs

In [2]:
import json
from pathlib import Path

import pandas as pd
from huggingface_hub import snapshot_download
from IPython.display import display

REPO_ID = "LLM-Digital-Twin/Twin-2K-500"
LOCAL_DIR = Path("hf")
CSV_DIR = LOCAL_DIR / "question_catalog_and_human_response_csv"

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR,
    allow_patterns=["question_catalog_and_human_response_csv/*"],
)
sorted(p.name for p in CSV_DIR.iterdir())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

['generate_catalog_and_csvs.py',
 'question_catalog.json',
 'question_catalog_README.md',
 'wave1_3_response.csv',
 'wave1_3_response_label.csv',
 'wave4_response.csv',
 'wave4_response_label.csv']

## 2. Question catalog

In [3]:
import textwrap

catalog_df = pd.DataFrame(json.loads((CSV_DIR / "question_catalog.json").read_text()))


def extract_selector(row):
    if row["QuestionType"] == "DB":
        if pd.notna(row["Settings"]):
            raise ValueError(f"{row['QuestionID']}: DB row has unexpected Settings: {row['Settings']}")
        return ""
    return row["Settings"]["Selector"]


def print_catalog_entry(q, keys, width=90):
    for k in keys:
        val = q[k]
        if k == "QuestionText":
            wrapped = "\n".join(
                textwrap.fill(line, width=width, initial_indent="    ", subsequent_indent="    ")
                for line in str(val).splitlines() or [""]
            )
            print(f"  {k}:\n{wrapped}")
        else:
            print(f"  {k}: {val}")


catalog_df["Selector"] = catalog_df.apply(extract_selector, axis=1)
catalog_df["n_csv_cols"] = catalog_df["csv_columns"].map(len)

print(f"{len(catalog_df)} catalog entries ({catalog_df['QuestionID'].nunique()} unique QuestionIDs) -> {catalog_df['n_csv_cols'].sum()} CSV columns")
display(
    catalog_df.groupby(["QuestionType", "Selector"], as_index=False)
    .agg(n_catalog_entries=("QuestionID", "count"), n_csv_cols=("n_csv_cols", "sum"))
)
catalog_df.head(2)

256 catalog entries (246 unique QuestionIDs) -> 760 CSV columns


,QuestionType,Selector,n_catalog_entries,n_csv_cols
0,DB,,14,0
1,MC,MAHR,1,4
2,MC,MAVR,20,80
3,MC,SAHR,25,25
4,MC,SAVR,129,129
5,Matrix,Bipolar,13,158
6,Matrix,Likert,23,290
7,Slider,HSLIDER,3,12
8,TE,FORM,4,38
9,TE,ML,3,3


,QuestionID,QuestionText,QuestionType,Options,Settings,BlockName,source,csv_columns,is_descriptive,Rows,Columns,RowsID,Statements,Range,StatementsID,Selector,n_csv_cols
0,QID11,Which part of the United States do you current...,MC,"[Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME...","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID11],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1
1,QID12,What is the sex that you were assigned at birth?,MC,"[Male, Female]","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID12],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1


## 3. Human responses: waves 1–3 and wave 4

In [5]:
w13_df = pd.read_csv(CSV_DIR / "wave1_3_response_label.csv", dtype=str)
w13_df["pid"] = w13_df["pid"].astype(int)
w13_df = w13_df.set_index("pid")

w4_df = pd.read_csv(CSV_DIR / "wave4_response_label.csv", dtype=str)
w4_df["pid"] = w4_df["pid"].astype(int)
w4_df = w4_df.set_index("pid")

print(f"waves 1-3: {w13_df.shape[0]:,} respondents x {w13_df.shape[1]} columns")
print(f"wave 4:    {w4_df.shape[0]:,} respondents x {w4_df.shape[1]} columns")

if not w13_df.index.sort_values().equals(w4_df.index.sort_values()):
    raise ValueError("Waves 1-3 and wave 4 contain different pids")
missing_in_w13 = sorted(set(w4_df.columns) - set(w13_df.columns))
if missing_in_w13:
    raise ValueError(f"Wave 4 columns without a waves 1-3 counterpart: {missing_in_w13}")

all_cat_cols = [c for cols in catalog_df["csv_columns"] for c in cols]
if len(all_cat_cols) != len(set(all_cat_cols)):
    raise ValueError("Duplicate column names in catalog csv_columns")
if set(all_cat_cols) != set(w13_df.columns):
    raise ValueError("Catalog csv_columns do not match w13_df columns")
print(f"Verified 1-to-1 bijection between catalog csv_columns and w13_df ({len(all_cat_cols)} columns).")

waves 1-3: 2,058 respondents x 760 columns
wave 4:    2,058 respondents x 126 columns
Verified 1-to-1 bijection between catalog csv_columns and w13_df (760 columns).


In [55]:
# 3a. MC — Single-select (SAVR, SAHR): 154 catalog entries -> 1 CSV column ({QID}) each, storing SelectedText
mc_single_df = catalog_df[(catalog_df["QuestionType"] == "MC") & (catalog_df["Selector"].isin(["SAVR", "SAHR"]))]
print(f"MC single-select: {len(mc_single_df)} entries -> {mc_single_df['n_csv_cols'].sum()} CSV columns")

for _, q in mc_single_df.iterrows():
    (col,) = q["csv_columns"]
    unexpected = set(w13_df[col].dropna().unique()) - set(q["Options"])
    if unexpected:
        raise ValueError(f"{col}: values not in catalog Options: {unexpected}")

ex_mc = mc_single_df[mc_single_df["QuestionID"] == "QID63"].iloc[0]
print("\nCatalog entry example (QID63):")
print_catalog_entry(ex_mc, ["QuestionID", "BlockName", "QuestionType", "Selector", "QuestionText", "Options", "csv_columns"])
print("\nResponse counts in w13_df['QID63'] (in catalog Options order):")
display(w13_df["QID63"].value_counts(dropna=False).reindex(ex_mc["Options"]).rename("n").to_frame())

MC single-select: 154 entries -> 154 CSV columns

Catalog entry example (QID63):
  QuestionID: QID63
  BlockName: Cognitive tests 
  QuestionType: MC
  Selector: SAVR
  QuestionText:
    Synonym: Select the word that is most nearly the same in meaning as CONCUR
  Options: ['acquiesce', 'extricate', 'divulge', 'concoct', 'ransack']
  csv_columns: ['QID63']

Response counts in w13_df['QID63'] (in catalog Options order):


,n
QID63,
acquiesce,1614
extricate,113
divulge,175
concoct,128
ransack,28


In [56]:
# 3b. MC — Multi-select (MAVR, MAHR): 21 catalog entries -> 4 CSV columns ({QID}_1..{QID}_4) each
# Each column i stores Options[i-1] if selected by the respondent, or NaN if not selected.
mc_multi_df = catalog_df[(catalog_df["QuestionType"] == "MC") & (catalog_df["Selector"].isin(["MAVR", "MAHR"]))]
print(f"MC multi-select: {len(mc_multi_df)} entries -> {mc_multi_df['n_csv_cols'].sum()} CSV columns")

for _, q in mc_multi_df.iterrows():
    for col, opt in zip(q["csv_columns"], q["Options"], strict=True):
        vals = set(w13_df[col].dropna().unique())
        if not vals <= {opt}:
            raise ValueError(f"{col}: expected only {opt!r} or NaN, got {vals}")

ex_mavr = mc_multi_df[mc_multi_df["QuestionID"] == "QID221"].iloc[0]
print("\nCatalog entry example (QID221 — Wason selection card task):")
print_catalog_entry(ex_mavr, ["QuestionID", "BlockName", "QuestionType", "Selector", "QuestionText", "Options", "csv_columns"])
display(w13_df[ex_mavr["csv_columns"]].head(5))
print("Non-null selection counts per option column:")
display(
    pd.DataFrame({
        "Option": ex_mavr["Options"],
        "selected_n": [w13_df[c].notna().sum() for c in ex_mavr["csv_columns"]],
    }, index=ex_mavr["csv_columns"])
)

MC multi-select: 21 entries -> 84 CSV columns

Catalog entry example (QID221 — Wason selection card task):
  QuestionID: QID221
  BlockName: Cognitive tests 
  QuestionType: MC
  Selector: MAHR
  QuestionText:
    Imagine you see a number of cards from a set of cards. Each card in the set has a
    capital letter on one side and a number on the other. Naturally, only one side is
    visible in each case. For the set of cards, a rule has been stated: If there is an A
    on the letter side of the card, then there is a 3 on the number side. Four cards were
    drawn. Below is the information visible for each card (letter or number). Which of the
    following card(s) would have to be turned over in order to test the truth or falsity
    of the rule?
  Options: ['A', 'F', '3', '7']
  csv_columns: ['QID221_1', 'QID221_2', 'QID221_3', 'QID221_4']


,QID221_1,QID221_2,QID221_3,QID221_4
pid,,,,
1,A,NaN,NaN,NaN
2,NaN,NaN,3,NaN
3,A,NaN,NaN,NaN
4,A,NaN,3,NaN
5,A,NaN,3,NaN


Non-null selection counts per option column:


,Option,selected_n
QID221_1,A,1854
QID221_2,F,153
QID221_3,3,1251
QID221_4,7,222


In [57]:
# 3c. Matrix — Likert (Selector == "Likert"): 23 catalog entries -> 290 CSV columns ({QID}_{RowID})
# Each csv_columns[i] corresponds to sub-question Rows[i], answered with one scale label from Columns.
likert_df = catalog_df[(catalog_df["QuestionType"] == "Matrix") & (catalog_df["Selector"] == "Likert")]
print(f"Matrix Likert: {len(likert_df)} entries -> {likert_df['n_csv_cols'].sum()} CSV columns")

for _, q in likert_df.iterrows():
    scale_set = set(q["Columns"])
    for col, _ in zip(q["csv_columns"], q["Rows"], strict=True):
        unexpected = set(w13_df[col].dropna().unique()) - scale_set
        if unexpected:
            raise ValueError(f"{col}: values not in catalog Columns: {unexpected}")

ex_likert = likert_df[likert_df["QuestionID"] == "QID159"].iloc[0]
print("\nCatalog entry example (QID159):")
print_catalog_entry(ex_likert, ["QuestionID", "BlockName", "QuestionType", "Selector", "QuestionText", "Rows", "Columns", "csv_columns"])
print("\nRow mapping and response counts for QID159:")
likert_counts = pd.DataFrame(
    {col: w13_df[col].value_counts().reindex(ex_likert["Columns"], fill_value=0) for col in ex_likert["csv_columns"]}
).T
likert_counts.insert(0, "Row", ex_likert["Rows"])
display(likert_counts)

Matrix Likert: 23 entries -> 290 CSV columns

Catalog entry example (QID159):
  QuestionID: QID159
  BlockName: Linda -no conjunction 
  QuestionType: Matrix
  Selector: Likert
  QuestionText:
    Linda is 31 years old, single, outspoken, and very bright. She majored in philosophy.
    As a student, she was deeply concerned with issues of discrimination and social
    justice, and also participated in anti-nuclear demonstrations. Please complete the
    statements below.
  Rows: ['It is ___ that Linda is a teacher in an elementary school', 'It is ___ that Linda works in a bookstore and takes Yoga classes', 'It is ___ that Linda is a bank teller']
  Columns: ['Extremely improbable', 'Very improbable', 'Somewhat probable', 'Moderately probable', 'Very probable', 'Extremely probable']
  csv_columns: ['QID159_1', 'QID159_2', 'QID159_3']

Row mapping and response counts for QID159:


,Row,Extremely improbable,Very improbable,Somewhat probable,Moderately probable,Very probable,Extremely probable
QID159_1,It is ___ that Linda is a teacher in an elemen...,65,160,378,251,144,31
QID159_2,It is ___ that Linda works in a bookstore and ...,83,137,336,289,156,28
QID159_3,It is ___ that Linda is a bank teller,179,418,300,88,34,10


In [60]:
likert_counts[['Extremely improbable',	'Very improbable',	'Somewhat probable',	'Moderately probable',	'Very probable',	'Extremely probable']].sum(1)

/tmp/ipykernel_214612/3050745315.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  likert_counts[['Extremely improbable',	'Very improbable',	'Somewhat probable',	'Moderately probable',	'Very probable',	'Extremely probable']].sum(1)


QID159_1    1029
QID159_2    1029
QID159_3    1029
dtype: int64

In [68]:
# Breakdown of all 23 Matrix Likert entries with q_type = "score" if len(q["Columns"]) > 2 else "choice"
likert_overview = []
for _, q in likert_df.iterrows():
    col0 = q["csv_columns"][0]
    q_type = "score" if len(q["Columns"]) > 2 else "choice"
    likert_overview.append({
        "QuestionID": q["QuestionID"],
        "BlockName": q["BlockName"].strip(),
        "q_type": q_type,
        "n_rows": len(q["Rows"]),
        "n_scale_pts": len(q["Columns"]),
        "n_answered_row0": int(w13_df[col0].notna().sum()),
        "Columns": q["Columns"],
    })
likert_overview_df = pd.DataFrame(likert_overview)
print(likert_overview_df.groupby("q_type").agg(n_matrices=("QuestionID", "count"), n_csv_cols=("n_rows", "sum")).to_string())
display(likert_overview_df)

for qid, label in [
    ("QID29", "Partially-labeled 9-point scale (score)"),
    ("QID125", "Partially-labeled 4-point symptom scale (score)"),
    ("QID27", "Binary ['FALSE', 'TRUE'] matrix (choice)"),
    ("QID203", "Binary ['red', 'green'] die-roll matrix (choice)"),
    ("QID198", "Binary ['1', '2'] card-shuffle matrix (choice)"),
]:
    q = likert_df[likert_df["QuestionID"] == qid].iloc[0]
    print(f"\n=== {qid}: {label} ===")
    print_catalog_entry(q, ["QuestionID", "BlockName", "QuestionText", "Columns"])
    sample_cols = q["csv_columns"][:3]
    sample_counts = pd.DataFrame(
        {c: w13_df[c].value_counts().reindex(q["Columns"], fill_value=0) for c in sample_cols}
    ).T
    sample_counts.insert(0, "Row", q["Rows"][:3])
    sample_counts["total_n"] = sample_counts[q["Columns"]].sum(axis=1)
    display(sample_counts)

        n_matrices  n_csv_cols
q_type                        
choice           3          29
score           20         261


,QuestionID,BlockName,q_type,n_rows,n_scale_pts,n_answered_row0,Columns
0,QID25,Personality,score,44,5,2058,"[Disagree strongly, Disagree a little, Neither..."
1,QID26,Personality,score,18,5,2058,"[Disagree strongly, Disagree a little, Neither..."
2,QID29,Personality,score,24,9,2058,"[Not Important to me 1, 2, 3, 4, Quite Importa..."
3,QID35,Personality,score,12,5,2058,"[Disagree strongly, Disagree a little, Neither..."
4,QID232,Personality,score,20,5,2058,"[Disagree strongly, Disagree a little, Neither..."
5,QID235,Personality,score,6,5,2058,"[Disagree strongly, Disagree a little, Neither..."
6,QID27,Personality,choice,13,2,2058,"[FALSE, TRUE]"
7,QID30,Personality,score,8,9,2058,"[Extremely inaccurate 1, 2, 3, 4, Neither inac..."
8,QID125,Personality,score,21,4,2058,"[Not at all 0, 1, 2, Severely - I barely could..."
9,QID233,Personality,score,16,5,2058,"[Disagree strongly, Disagree a little, Neither..."



=== QID29: Partially-labeled 9-point scale (score) ===
  QuestionID: QID29
  BlockName: Personality 
  QuestionText:
    Below are 24 different values that people rate of different importance in their lives.
    FIRST READ THROUGH THE LIST to familiarize yourself with all the values. While reading
    over the list, consider which ones tend to be most important to you and which tend to
    be least important to you. After familiarizing yourself with the list, rate the
    relative importance of each value to you as “A GUIDING PRINCIPLE IN MY LIFE.” It is
    important to spread your ratings out as best you can—be sure to use some numbers in
    the lower range, some in the middle range, and some in the higher range. Avoid using
    too many similar numbers. Work fairly quickly.
  Columns: ['Not Important to me 1', '2', '3', '4', 'Quite Important to me 5', '6', '7', '8', 'Highly Important to me 9']


,Row,Not Important to me 1,2,3,4,Quite Important to me 5,6,7,8,Highly Important to me 9,total_n
QID29_1,"WEALTH (financially successful, prosperous)",111,186,250,240,296,213,300,228,234,2058
QID29_2,PLEASURE (having one’s fill of life’s pleasure...,13,35,78,134,265,283,437,393,420,2058
QID29_3,"FORGIVENESS (pardoning others’ faults, being m...",67,99,129,178,323,298,318,322,324,2058



=== QID125: Partially-labeled 4-point symptom scale (score) ===
  QuestionID: QID125
  BlockName: Personality 
  QuestionText:
    How much have you been bothered by each of the following symptoms over the past week?
  Columns: ['Not at all 0', '1', '2', 'Severely - I barely could stand it 3']


,Row,Not at all 0,1,2,Severely - I barely could stand it 3,total_n
QID125_1,Numbness or tingling,1441,417,172,28,2058
QID125_2,Feeling hot,1124,560,330,44,2058
QID125_3,Wobbliness in legs,1608,314,117,19,2058


... (2 outputs left out)


=== QID203: Binary ['red', 'green'] die-roll matrix (choice) ===
  QuestionID: QID203
  BlockName: Probability matching vs. maximizing - Problem 2
  QuestionText:
    Consider the following situation: A die with 4 red faces and 2 green faces will be
    rolled 6 times. Before each roll you will be asked to predict which color (red or
    green) will show up once the die is rolled. Which color is most likely to show up
    after ...
  Columns: ['red', 'green']


,Row,red,green,total_n
QID203_1,roll #1,981,45,1026
QID203_2,roll #2,879,147,1026
QID203_3,roll #3,692,334,1026



=== QID198: Binary ['1', '2'] card-shuffle matrix (choice) ===
  QuestionID: QID198
  BlockName: Probability matching vs. maximizing - Problem 1
  QuestionText:
    Consider the following hypothetical situation: A deck with 10 cards is randomly
    shuffled 10 separate times. The 10 cards are composed of 7 cards with the number “1”
    on the down side and 3 cards with the number “2” on the down side. Each time the 10
    cards are reshuffled, your task is to predict the number on the down side of the top
    card. Imagine that you will receive $100 for each downside number you correctly
    predict, and that you want to earn as much money as possible. What would you predict
    after ...
  Columns: ['1', '2']


,Row,1,2,total_n
QID198_1,shuffle #1,952,80,1032
QID198_2,shuffle #2,887,145,1032
QID198_3,shuffle #3,773,259,1032


In [69]:
# 3d. Matrix — Bipolar (Selector == "Bipolar"): 13 catalog entries -> 158 CSV columns ({QID}_{RowID})
# Each row becomes a binary 'choice' question keeping QuestionText verbatim as instructions
# and splitting Rows[i] on ':' into opt1 ('1') and opt2 ('2').
bipolar_df = catalog_df[(catalog_df["QuestionType"] == "Matrix") & (catalog_df["Selector"] == "Bipolar")]
print(f"Matrix Bipolar: {len(bipolar_df)} entries -> {bipolar_df['n_csv_cols'].sum()} CSV columns (all q_type='choice')")

bipolar_overview = []
for _, q in bipolar_df.iterrows():
    if q["Columns"] != ["1", "2"]:
        raise ValueError(f"{q['QuestionID']}: expected Columns ['1', '2'], got {q['Columns']}")
    for col, row_str in zip(q["csv_columns"], q["Rows"], strict=True):
        if row_str.count(":") != 1:
            raise ValueError(f"{col}: expected '<left>:<right>' in Rows, got {row_str!r}")
        unexpected = set(w13_df[col].dropna().unique()) - {"1", "2"}
        if unexpected:
            raise ValueError(f"{col}: unexpected Bipolar response values: {unexpected}")
    bipolar_overview.append({
        "QuestionID": q["QuestionID"],
        "n_rows": len(q["Rows"]),
        "first_row": q["Rows"][0],
        "last_row": q["Rows"][-1],
        "QuestionText_start": q["QuestionText"][:75] + "...",
    })
display(pd.DataFrame(bipolar_overview))

ex_bip = bipolar_df[bipolar_df["QuestionID"] == "QID84"].iloc[0]
print("\nCatalog entry example (QID84 — time discounting):")
print_catalog_entry(ex_bip, ["QuestionID", "BlockName", "QuestionType", "Selector", "QuestionText", "Rows", "RowsID", "Columns", "csv_columns"])

bip_summary = []
for col, row_str in zip(ex_bip["csv_columns"], ex_bip["Rows"], strict=True):
    opt1, opt2 = row_str.split(":")
    bip_summary.append({
        "csv_column": col,
        "Option 1 (left)": opt1,
        "Option 2 (right)": opt2,
        "n_1": int((w13_df[col] == "1").sum()),
        "n_2": int((w13_df[col] == "2").sum()),
    })
display(pd.DataFrame(bip_summary).set_index("csv_column"))

Matrix Bipolar: 13 entries -> 158 CSV columns (all q_type='choice')


,QuestionID,n_rows,first_row,last_row,QuestionText_start
0,QID84,9,$6.00 in 6 weeks:$3.00 in 5 weeks,$6.00 in 6 weeks:$7.00 in 5 weeks,Please choose between the following options. F...
1,QID244,10,$8.00 in 7 weeks:$4.00 in 6 weeks,$8.00 in 7 weeks:$9.00 in 6 weeks,Please choose between the following options. F...
2,QID245,11,$10.00 in 7 weeks:$5.00 in 5 weeks,$10.00 in 7 weeks:$11.00 in 5 weeks,Please choose between the following options. F...
3,QID246,9,$6.00 in 1 week:$3.00 now,$6.00 in 1 week:$7.00 now,Please choose between the following options. F...
4,QID247,10,$8.00 in 1 week:$4.00 now,$8.00 in 1 week:$9.00 now,Please choose between the following options. F...
5,QID248,11,$10.00 in 2 weeks:$5.00 now,$10.00 in 2 weeks:$11.00 now,Please choose between the following options. F...
6,QID250,14,Lottery:$0.50,Lottery:$5.00,"In this question, the LOTTERY is a 50% chance ..."
7,QID251,14,Lottery:$2.50,Lottery:$7.00,"In this question, the LOTTERY is a 50% chance ..."
8,QID252,14,Lottery:$2.50,Lottery:$7.00,"In this question, the LOTTERY is a 50% chance ..."
9,QID276,14,Lottery:-$5.50,Lottery:-$1.00,"In this question, the LOTTERY is a 50% chance ..."



Catalog entry example (QID84 — time discounting):
  QuestionID: QID84
  BlockName: Economic preferences 
  QuestionType: Matrix
  Selector: Bipolar
  QuestionText:
    Please choose between the following options. For each line in the list, you must
    choose between the option on the left and the option on the right. Note that on each
    line, the option on the left stays the same while the option on the right gets better
    as one goes down the list. You can select the option you would prefer receiving by
    clicking on the button next to that option.
  Rows: ['$6.00 in 6 weeks:$3.00 in 5 weeks', '$6.00 in 6 weeks:$4.00 in 5 weeks', '$6.00 in 6 weeks:$4.50 in 5 weeks', '$6.00 in 6 weeks:$5.00 in 5 weeks', '$6.00 in 6 weeks:$5.25 in 5 weeks', '$6.00 in 6 weeks:$5.50 in 5 weeks', '$6.00 in 6 weeks:$5.75 in 5 weeks', '$6.00 in 6 weeks:$6.00 in 5 weeks', '$6.00 in 6 weeks:$7.00 in 5 weeks']
  RowsID: ['4', '5', '6', '7', '8', '9', '10', '11', '12']
  Columns: ['1', '2']
  csv_columns

,Option 1 (left),Option 2 (right),n_1,n_2
csv_column,,,,
QID84_4,$6.00 in 6 weeks,$3.00 in 5 weeks,1984,74
QID84_5,$6.00 in 6 weeks,$4.00 in 5 weeks,1952,106
QID84_6,$6.00 in 6 weeks,$4.50 in 5 weeks,1921,137
QID84_7,$6.00 in 6 weeks,$5.00 in 5 weeks,1654,404
QID84_8,$6.00 in 6 weeks,$5.25 in 5 weeks,1548,510
QID84_9,$6.00 in 6 weeks,$5.50 in 5 weeks,1389,669
QID84_10,$6.00 in 6 weeks,$5.75 in 5 weeks,1239,819
QID84_11,$6.00 in 6 weeks,$6.00 in 5 weeks,140,1918
QID84_12,$6.00 in 6 weeks,$7.00 in 5 weeks,61,1997


In [73]:
# 3e. Slider (Selector == "HSLIDER"): 3 catalog entries -> 12 CSV columns ({QID} or {QID}_{StatementsID})
# Excluded (q_type='exclude'): responses are continuous integers in [0, 100] (81-100 unique values per column).
slider_df = catalog_df[catalog_df["QuestionType"] == "Slider"]
print(f"Slider: {len(slider_df)} entries -> {slider_df['n_csv_cols'].sum()} CSV columns (all q_type='exclude')")

for _, q in slider_df.iterrows():
    print(f"\nCatalog entry ({q['QuestionID']}):")
    print_catalog_entry(q, ["QuestionID", "BlockName", "QuestionType", "Selector", "QuestionText", "Statements", "Range", "csv_columns"])

slider_rows = []
for _, q in slider_df.iterrows():
    rmin, rmax = q["Range"]["Min"], q["Range"]["Max"]
    for col, stmt in zip(q["csv_columns"], q["Statements"], strict=True):
        vals = pd.to_numeric(w13_df[col].dropna(), errors="raise")
        if (vals < rmin).any() or (vals > rmax).any():
            raise ValueError(f"{col}: values outside [{rmin}, {rmax}]")
        slider_rows.append({
            "QuestionID": q["QuestionID"],
            "csv_column": col,
            "q_type": "exclude",
            "Range": f"[{rmin}, {rmax}]",
            "Statement": stmt if stmt else "(single slider)",
            "n_non_null": len(vals),
            "n_unique": vals.nunique(),
            "mean": round(vals.mean(), 2),
        })
display(pd.DataFrame(slider_rows).set_index("csv_column"))

Slider: 3 entries -> 12 CSV columns (all q_type='exclude')

Catalog entry (QID154):
  QuestionID: QID154
  BlockName: Base-rate 30 engineers 
  QuestionType: Slider
  Selector: HSLIDER
  QuestionText:
    A panel of psychologist have interviewed and administered personality tests to 30
    engineers and 70 lawyers, all successful in their respective fields. On the basis of
    this information, thumbnail descriptions of the 30 engineers and 70 lawyers have been
    written. Below is one description, chosen at random from the 100 available
    descriptions. Jack is a 45-year-old man. He is married and has four children. He is
    generally conservative, careful, and ambitious. He shows no interest in political and
    social issues and spends most of his free time on his many hobbies which include home
    carpentry, sailing, and mathematical puzzles. The probability that Jack is one of the
    30 engineers in the sample of 100 is ___%. Please indicate the probability on a scale
    fro

,QuestionID,q_type,Range,Statement,n_non_null,n_unique,mean
csv_column,,,,,,,
QID154,QID154,exclude,"[0, 100]",(single slider),1043,90,52.17
QID290_1,QID290,exclude,"[0, 100]",Placing a tax on carbon emissions,2058,100,47.18
QID290_2,QID290,exclude,"[0, 100]",Ensuring 40% of all new clean energy infrastru...,2058,99,41.55
QID290_3,QID290,exclude,"[0, 100]",Federal investments to ensure a carbon-polluti...,2058,99,45.71
QID290_4,QID290,exclude,"[0, 100]",A ‘Medicare for All’ system in which all Ameri...,2058,100,54.99
QID290_5,QID290,exclude,"[0, 100]","A ‘public option’, which would allow Americans...",2058,99,56.48
QID290_6,QID290,exclude,"[0, 100]",Immigration reforms that would provide a path ...,2058,100,45.62
QID290_7,QID290,exclude,"[0, 100]",A law that requires companies to provide paid ...,2058,98,66.14
QID290_10,QID290,exclude,"[0, 100]",A 2% tax on the assets of individuals with a n...,2058,100,61.94


In [74]:
# 3f. TE — Text Entry (SL, ML, FORM): 28 catalog entries -> 62 CSV columns ({QID}_TEXT or {QID}_{RowID})
# and 3g. DB — Descriptive Block: 14 entries -> 0 CSV columns (both q_type='exclude')
te_df = catalog_df[catalog_df["QuestionType"] == "TE"].copy()
te_df["ContentType"] = te_df["Settings"].apply(lambda s: s["ContentType"])
print(f"TE (Text Entry): {len(te_df)} entries -> {te_df['n_csv_cols'].sum()} CSV columns (all q_type='exclude')")
te_summary = (
    te_df.groupby(["Selector", "ContentType"], as_index=False)
    .agg(n_entries=("QuestionID", "count"), n_csv_cols=("n_csv_cols", "sum"))
)
te_summary["q_type"] = "exclude"
display(te_summary)

for sel, ct in [("SL", "ValidNumber"), ("SL", ""), ("ML", ""), ("FORM", "")]:
    ex_te = te_df[(te_df["Selector"] == sel) & (te_df["ContentType"] == ct)].iloc[0]
    col0 = ex_te["csv_columns"][0]
    print(f"\nTE example (Selector={sel!r}, ContentType={ct!r}):")
    print_catalog_entry(ex_te, ["QuestionID", "BlockName", "QuestionType", "Selector", "QuestionText", "csv_columns"])
    print(f"  Sample responses in {col0}: {w13_df[col0].dropna().head(4).tolist()}")

db_df = catalog_df[catalog_df["QuestionType"] == "DB"]
if (db_df["n_csv_cols"] != 0).any():
    raise ValueError("DB entries must have 0 CSV columns")
print(f"\nDB (Descriptive Block): {len(db_df)} instructional entries -> {db_df['n_csv_cols'].sum()} CSV columns (all q_type='exclude')")
print_catalog_entry(db_df.iloc[0], ["QuestionID", "BlockName", "QuestionType", "QuestionText", "csv_columns"])

TE (Text Entry): 28 entries -> 62 CSV columns (all q_type='exclude')


,Selector,ContentType,n_entries,n_csv_cols,q_type
0,FORM,,4,38,exclude
1,ML,,3,3,exclude
2,SL,,2,2,exclude
3,SL,ValidNumber,19,19,exclude



TE example (Selector='SL', ContentType='ValidNumber'):
  QuestionID: QID53
  BlockName: Cognitive tests 
  QuestionType: TE
  Selector: SL
  QuestionText:
    How many cubic feet of dirt are there in a hole that is 3’ deep x 3’ wide x 3’ long?
    Enter a number of cubic feet.
  csv_columns: ['QID53_TEXT']
  Sample responses in QID53_TEXT: ['27', '27', '9', '0']

TE example (Selector='SL', ContentType=''):
  QuestionID: QID52
  BlockName: Cognitive tests 
  QuestionType: TE
  Selector: SL
  QuestionText:
    Emily’s father has three daughters. The first two are named April and May. What is the
    third daughter’s name?
  csv_columns: ['QID52_TEXT']
  Sample responses in QID52_TEXT: ['emily', 'Emily', 'Emily', 'Emily']

TE example (Selector='ML', ContentType=''):
  QuestionID: QID268
  BlockName: Personality 
  QuestionType: TE
  Selector: ML
  QuestionText:
    In this task we would like you to write about the type of person you aspire to be vs.
    the person you ought to be vs. the

## 4. Demographics

In [6]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np

DATA_DIR = Path.cwd().parent if Path.cwd().parent.name == "data" else Path.cwd().parent / "data"
CODE_DIR = DATA_DIR.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from data import persona

demo_df = catalog_df.loc[catalog_df["BlockName"] == "Demographics", ["QuestionID", "QuestionText", "Options", "csv_columns"]]
demo_by_qid = demo_df.set_index("QuestionID")

# Load respondent-level ISSP US personas (2015-2024) and SubPOP evaluation groups for side-by-side comparison
issp_resp_frames = []
for p in sorted(DATA_DIR.glob("issp*/issp*_US_long.parquet")):
    df_year = pd.read_parquet(p).drop_duplicates(subset=["CASEID"]).copy()
    df_year["year"] = p.parent.name
    issp_resp_frames.append(df_year)
issp_resp_df = pd.concat(issp_resp_frames, ignore_index=True)

subpop_groups = {}
with (DATA_DIR / "subpop/subpop_eval.jsonl").open() as f:
    for line in f:
        b = json.loads(line)
        for k, v in b["persona"].items():
            if k != "Country" and v not in subpop_groups.setdefault(k, []):
                subpop_groups[k].append(v)

print(f"Twin-2K-500 respondents: {len(w13_df):,} | ISSP US respondents (2015-2024): {len(issp_resp_df):,}")
display(demo_df)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Twin-2K-500 respondents: 2,058 | ISSP US respondents (2015-2024): 14,356


,QuestionID,QuestionText,Options,csv_columns
0,QID11,Which part of the United States do you current...,"[Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME...",[QID11]
1,QID12,What is the sex that you were assigned at birth?,"[Male, Female]",[QID12]
2,QID13,How old are you?,"[18-29, 30-49, 50-64, 65+]",[QID13]
3,QID14,What is the highest level of schooling or degr...,"[Less than high school, High school graduate, ...",[QID14]
4,QID15,What is your race or origin?,"[White, Black, Asian, Hispanic, Other]",[QID15]
5,QID16,Are you a citizen of the United States?,"[Yes, No]",[QID16]
6,QID17,Which of these best describes you?,"[Married, Living with a partner, Divorced, Sep...",[QID17]
7,QID18,"What is your present religion, if any?","[Protestant, Roman Catholic, Mormon, Orthodox,...",[QID18]
8,QID19,"Aside from weddings and funerals, how often do...","[More than once a week, Once a week, Once or t...",[QID19]
9,QID20,"In politics today, do you consider yourself a","[Republican, Democrat, Independent, Something ...",[QID20]


In [7]:
# 4.1 SEX (QID12) and AGE (QID13) — mapped via code/data/persona.py
print(f"=== SEX ({persona.PERSONA_NAMES['SEX']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['SEX']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID12"], "SEX", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["SEX"], "SEX", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["SEX"] + [np.nan]))

print(f"=== AGE ({persona.PERSONA_NAMES['AGE']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['AGE']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID13"], "AGE", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["AGE"], "AGE", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["AGE"] + [np.nan]))

=== SEX (Sex) | Global levels: ['Male', 'Female'] ===


,Twin-2K-500,ISSP (2015-2024)
SEX,,
Male,1014.0,6425
Female,1044.0,7901
NaN,NaN,30


=== AGE (Age) | Global levels: ['18-29', '30-49', '50-64', '65+'] ===


,Twin-2K-500,ISSP (2015-2024)
AGE,,
18-29,388.0,2030
30-49,735.0,4810
50-64,658.0,3580
65+,277.0,3520
NaN,NaN,416


In [8]:
# 4.2 US_REG (QID11) — mapped via code/data/persona.py
print(f"=== US_REG ({persona.PERSONA_NAMES['US_REG']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['US_REG']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID11"], "US_REG", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["US_REG"], "US_REG", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["US_REG"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["US_REG"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["US_REG"]))

=== US_REG (Region) | Global levels: ['Northeast', 'Midwest', 'South', 'West'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Northeast,342,2264,True
Midwest,372,3405,False
South,834,5534,True
West,510,3153,False


In [11]:
# 4.3 EDULEVEL (QID14) — mapped via code/data/persona.py
print(f"=== EDULEVEL ({persona.PERSONA_NAMES['EDULEVEL']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['EDULEVEL']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID14"], "EDULEVEL", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["EDULEVEL"], "EDULEVEL", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["EDULEVEL"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["EDULEVEL"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["EDULEVEL"] + [np.nan]))

=== EDULEVEL (Education) | Global levels: ['Less than high school', 'High school or some college', "Associate's degree", 'College graduate/some postgrad'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Less than high school,17.0,1384,True
High school or some college,740.0,6456,False
Associate's degree,253.0,1250,False
College graduate/some postgrad,1048.0,5224,True
NaN,NaN,42,NaN


In [16]:
# 4.4 RACE (QID15) and US_ETHN1 (ISSP Ancestry) — kept separate in code/data/persona.py
print(f"=== RACE ({persona.PERSONA_NAMES['RACE']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['RACE']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID15"], "RACE", "twin2k").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["RACE"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["RACE"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["RACE"]))

print(f"=== US_ETHN1 ({persona.PERSONA_NAMES['US_ETHN1']}) in ISSP (top 10 of {issp_resp_df['US_ETHN1'].nunique()} ancestry labels) ===")
display(issp_resp_df["US_ETHN1"].value_counts(dropna=False).head(10).to_frame("ISSP (2015-2024) n"))

=== RACE (Race) | Global levels: ['White', 'Black', 'Asian', 'Hispanic', 'Other'] ===


,Twin-2K-500,in_SubPOP_eval
White,1361,True
Black,251,True
Asian,140,True
Hispanic,194,True
Other,112,False


=== US_ETHN1 (Ancestry) in ISSP (top 10 of 96 ancestry labels) ===


,ISSP (2015-2024) n
US_ETHN1,
Germany,1923
England and Wales,1916
NaN,1815
Ireland,1220
Mexico,806
Other African,647
Italy,592
American Indian,503
Scotland,364


In [20]:
# 4.5 QID16 (US citizenship — excluded from persona) and MARITAL (QID17) — mapped via code/data/persona.py
print(f"=== MARITAL ({persona.PERSONA_NAMES['MARITAL']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['MARITAL']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID17"], "MARITAL", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["MARITAL"], "MARITAL", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["MARITAL"] + [np.nan]))

=== MARITAL (Marital status) | Global levels: ['Married', 'Living with a partner', 'Divorced', 'Separated', 'Widowed', 'Never been married'] ===


,Twin-2K-500,ISSP (2015-2024)
MARITAL,,
Married,813.0,6281.0
Living with a partner,212.0,NaN
Divorced,218.0,2389.0
Separated,31.0,422.0
Widowed,70.0,1137.0
Never been married,714.0,4090.0
NaN,NaN,37.0


In [24]:
# 4.6 RELIGGRP (QID18) — mapped via code/data/persona.py
print(f"=== RELIGGRP ({persona.PERSONA_NAMES['RELIGGRP']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['RELIGGRP']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID18"], "RELIGGRP", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["RELIGGRP"], "RELIGGRP", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["RELIGGRP"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["RELIGGRP"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["RELIGGRP"] + [np.nan]))

=== RELIGGRP (Religion) | Global levels: ['Protestant', 'Catholic', 'Orthodox', 'Other Christian', 'Jewish', 'Muslim', 'Buddhist', 'Hindu', 'Atheist', 'Agnostic', 'Nothing in particular', 'Other'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Protestant,510.0,6276.0,True
Catholic,358.0,3033.0,False
Orthodox,17.0,87.0,False
Other Christian,15.0,298.0,False
Jewish,39.0,250.0,True
Muslim,18.0,107.0,True
Buddhist,25.0,123.0,False
Hindu,7.0,96.0,True
Atheist,216.0,3635.0,True
Agnostic,311.0,NaN,False


In [28]:
# 4.7 ATTEND (QID19) — mapped via code/data/persona.py
print(f"=== ATTEND ({persona.PERSONA_NAMES['ATTEND']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['ATTEND']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID19"], "ATTEND", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["ATTEND"], "ATTEND", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["ATTEND"] + [np.nan]))

=== ATTEND (Attendance of religious services) | Global levels: ['More than once a week', 'Once a week', 'Once or twice a month', 'A few times a year', 'Seldom', 'Never'] ===


,Twin-2K-500,ISSP (2015-2024)
ATTEND,,
More than once a week,87.0,741
Once a week,295.0,2132
Once or twice a month,129.0,2440
A few times a year,246.0,3255
Seldom,463.0,1429
Never,838.0,4261
NaN,NaN,98


In [30]:
# 4.8 US_PRTY (QID20) — union of levels in code/data/persona.py
print(f"=== US_PRTY ({persona.PERSONA_NAMES['US_PRTY']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['US_PRTY']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID20"], "US_PRTY", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["US_PRTY"], "US_PRTY", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["US_PRTY"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["US_PRTY"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["US_PRTY"] + [np.nan]))

=== US_PRTY (Political party) | Global levels: ['Democrat', 'Republican', 'Independent', 'Something else', 'Other candidate'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Democrat,847.0,5174.0,True
Republican,540.0,3514.0,True
Independent,609.0,NaN,False
Something else,62.0,NaN,False
Other candidate,NaN,427.0,False
NaN,NaN,5241.0,NaN


In [35]:
# 4.9 US_INC (QID21) — mapped via code/data/persona.py
inc_levels = ["Less than $30,000", "$30,000 - $50,000", "$50,000 - $75,000", "$75,000 - $89,999", "$75,000 - $100,000", "$90,000 - $109,999", "$100,000 or more"]
print(f"=== US_INC ({persona.PERSONA_NAMES['US_INC']}) | Global levels (ISSP/Twin/SubPOP): {inc_levels} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID21"], "US_INC", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["US_INC"], "US_INC", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["US_INC"]["subpop"].values() for r in inc_levels}),
}).reindex(inc_levels + [np.nan]))

=== US_INC (Income) | Global levels (ISSP/Twin/SubPOP): ['Less than $30,000', '$30,000 - $50,000', '$50,000 - $75,000', '$75,000 - $89,999', '$75,000 - $100,000', '$90,000 - $109,999', '$100,000 or more'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
"Less than $30,000",367.0,3442.0,True
"$30,000 - $50,000",412.0,2119.0,False
"$50,000 - $75,000",411.0,2302.0,False
"$75,000 - $89,999",NaN,1051.0,False
"$75,000 - $100,000",316.0,NaN,False
"$90,000 - $109,999",NaN,1027.0,False
"$100,000 or more",552.0,2968.0,True
NaN,NaN,1447.0,NaN


In [39]:
# 4.10 PARTY_LR (QID22) — union of levels in code/data/persona.py
party_lr_levels = ["Very liberal", "Liberal", "Moderate", "Conservative", "Very conservative", "Other"]
print(f"=== PARTY_LR ({persona.PERSONA_NAMES['PARTY_LR']}) | Global levels (ISSP/Twin/SubPOP): {party_lr_levels} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID22"], "PARTY_LR", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["PARTY_LR"], "PARTY_LR", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["PARTY_LR"]["subpop"].values() for r in party_lr_levels}),
}).reindex(party_lr_levels + [np.nan]))

=== PARTY_LR (Political ideology) | Global levels (ISSP/Twin/SubPOP): ['Very liberal', 'Liberal', 'Moderate', 'Conservative', 'Very conservative', 'Other'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Very liberal,345.0,NaN,False
Liberal,564.0,5193.0,True
Moderate,582.0,NaN,True
Conservative,430.0,3514.0,True
Very conservative,137.0,NaN,False
Other,NaN,408.0,False
NaN,NaN,5241.0,NaN


In [42]:
# 4.11 QID23 (Household size)
print("=== Twin-2K-500 QID23 (Household size — raw) ===")
display(w13_df["QID23"].value_counts(dropna=False).reindex(demo_by_qid.loc["QID23", "Options"]).to_frame("Twin-2K-500 n"))

=== Twin-2K-500 QID23 (Household size — raw) ===


,Twin-2K-500 n
QID23,
1,412
2,650
3,423
4,352
More than 4,221


## 5. Pricing study: purchase share per product

In [41]:
pricing_df = catalog_df[
    (catalog_df["BlockName"] == "Product Preferences - Pricing") & (catalog_df["QuestionType"] == "MC")
]
PRICING_COLS = [c for cols in pricing_df["csv_columns"] for c in cols]
PRICING_OPTIONS = {"Yes, I would purchase the product", "No, I would not purchase the product"}
print(f"{len(PRICING_COLS)} pricing items")

for name, df in [("waves 1-3", w13_df), ("wave 4", w4_df)]:
    values = set(pd.unique(df[PRICING_COLS].to_numpy().ravel()))
    if values != PRICING_OPTIONS:
        raise ValueError(f"{name}: pricing answers {values} != {PRICING_OPTIONS}")

product_text = pricing_df.set_index(pricing_df["csv_columns"].str[0])["QuestionText"].str.extract(
    r"see the following product in that category: (.*)\. The product is priced at"
)[0]
purchase_share_df = pd.DataFrame({
    "product": product_text,
    "yes_wave1_3": (w13_df[PRICING_COLS] == "Yes, I would purchase the product").mean(),
    "yes_wave4": (w4_df[PRICING_COLS] == "Yes, I would purchase the product").mean(),
})
display(purchase_share_df.round(3))

40 pricing items


,product,yes_wave1_3,yes_wave4
QID9_1,"Land O Lakes Salted Stick Butter, 16 oz, 4 Sticks",0.433,0.444
QID9_2,Tylenol Extra Strength Caplets with 500 mg Ace...,0.431,0.443
QID9_3,Purina Fancy Feast Chicken Feast Classic Grain...,0.452,0.457
QID9_4,Smart Ones Three Cheese Ziti Marinara Frozen M...,0.441,0.458
QID9_5,Goya Cooked Ham 16 oz,0.451,0.455
QID9_6,"Lay's Classic Potato Snack Chips, Party Size, ...",0.431,0.449
QID9_7,"OZARKA Brand 100% Natural Spring Water, 16.9-o...",0.446,0.453
QID9_8,"Purina Dog Chow Complete, Dry Dog Food for Adu...",0.439,0.436
QID9_9,"Fresh Raspberries, 12 oz Container",0.431,0.429
QID9_10,Oscar Mayer Chopped Ham & Water product Deli L...,0.445,0.451


## 6. Per-respondent prices: `wave_split` JSON

In [ ]:
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR,
    allow_patterns=["wave_split/*"],
)
wave_split_df = pd.read_parquet(LOCAL_DIR / "wave_split" / "chunks")
print(f"{len(wave_split_df):,} rows; columns: {list(wave_split_df.columns)}")

record = json.loads(wave_split_df.iloc[0]["wave4_Q_wave4_A"])
print(type(record))
print(json.dumps(record, indent=1)[:4000])